# signVLM training (Google Colab)

This notebook follows the **SignVLM** stack in this repository (`main.py`, `model.py`, `video_dataset/`): CLIP ViT backbone + temporal decoder + classifier, **AdamW**, **CosineAnnealingLR**, **CrossEntropyLoss**, **AMP fp16**, and list files in the form `path<TAB>label` (see `SIGNVLM_TRAINING_NOTES.md` and `docs/signVLMPipeline.md`).

**Cell 1 — Setup:** installs PyAV (`av`), **scikit-learn** (confusion matrix + F1), **pillow**, and prints PyTorch/CUDA.

In [1]:
# Cell 1: Setup and installations
import subprocess
import sys

def pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install(["av", "pillow", "scikit-learn"])
print("OK: av (PyAV), pillow, scikit-learn")

import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

OK: av (PyAV), pillow, scikit-learn
torch: 2.10.0+cu128 cuda: True


## Cell 2: Mount Drive and paths

`BASE` must stay under **`/content/drive/MyDrive/...`** so **checkpoints** (`signVLM_checkpoints/`) and **list files** (`signVLM_lists/`) are stored on **Google Drive**, not Colab’s ephemeral `/content` disk.

Expected layout under `BASE`:

- `train_data/<class_name>/*.mp4` (or frames under `<class>/<stem>/` if using `frames_available=1`)
- `validation_data/...`
- `test_data/...`
- `Unseen_data/...`

Place this **repository** on Drive (or clone it there); `REPO_ROOT` defaults next to `FYP_DATA`.

In [ ]:
# Cell 2: Mount Google Drive and directory paths
# Training weights are written by checkpoint.save_checkpoint() to CHECKPOINT_DIR below.
# That path is under the mounted Drive — NOT under /content alone (runtime is wiped when the session ends).
from google.colab import drive
drive.mount("/content/drive")

import os

DRIVE_MY = "/content/drive/MyDrive"
BASE = f"{DRIVE_MY}/FYP_DATA"
TRAIN_DIR = f"{BASE}/train_data"
VAL_DIR = f"{BASE}/validation_data"
TEST_DIR = f"{BASE}/test_data"
UNSEEN_DIR = f"{BASE}/Unseen_data"

# Repository root — default folder name; auto-search under BASE if missing (nested zip / rename)
REPO_ROOT = f"{BASE}/Urdu-Sign-Language-Recognition-using-SignVLM"


def find_signvlm_repo_under(start: str, max_depth: int = 6):
    """Find a directory that looks like this repo: checkpoint.py, main.py, video_dataset/."""
    start = os.path.abspath(start)
    if not os.path.isdir(start):
        return None
    for root, dirs, files in os.walk(start):
        rel = root[len(start) :].strip(os.sep)
        depth = 0 if not rel else rel.count(os.sep) + 1
        if depth > max_depth:
            dirs.clear()
            continue
        names_lower = {n.lower() for n in files}
        if "checkpoint.py" not in names_lower or "main.py" not in names_lower:
            continue
        if os.path.isdir(os.path.join(root, "video_dataset")):
            return root
    return None


def _folder_has_checkpoint_py(folder: str) -> bool:
    if not os.path.isdir(folder):
        return False
    try:
        for name in os.listdir(folder):
            if name.lower() == "checkpoint.py" and os.path.isfile(os.path.join(folder, name)):
                return True
    except OSError:
        pass
    return False


if not _folder_has_checkpoint_py(REPO_ROOT):
    found = find_signvlm_repo_under(BASE)
    if found:
        REPO_ROOT = found
        print("Auto-detected REPO_ROOT under BASE:", REPO_ROOT)

# CLIP ViT-L/14 weights (OpenAI JIT .pt); adjust if yours live elsewhere
BACKBONE_PATH = f"{REPO_ROOT}/CLIP_weights/ViT-L/ViT-L/ViT-L-14.pt"

# Persist checkpoints + TSV lists on Google Drive (same tree as your dataset)
CHECKPOINT_DIR = f"{BASE}/signVLM_checkpoints"
LIST_DIR = f"{BASE}/signVLM_lists"

if not CHECKPOINT_DIR.startswith("/content/drive/MyDrive"):
    raise RuntimeError(
        "CHECKPOINT_DIR must be under mounted Google Drive (/content/drive/MyDrive/...). "
        "Do not set BASE to a /content path only — you would lose weights after disconnect."
    )

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LIST_DIR, exist_ok=True)
print("Dataset / project BASE:", BASE)
print("REPO_ROOT:", REPO_ROOT)
print("Weights & optimizer checkpoints (Google Drive):", CHECKPOINT_DIR)
print("List files (Google Drive):", LIST_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Cell 3: Imports

Run **Cell 2** first so `REPO_ROOT` points at the **repository root** (the folder that contains `checkpoint.py` — this is local project code, not `pip install`). The cell resolves that path and adds it to `sys.path`, then initializes **single-GPU distributed** like `main.py` (FileStore + `gloo`).

In [9]:
# Cell 3: Imports and single-process distributed init (matches main.py)
# `checkpoint` is this repo's checkpoint.py — not a pip package. REPO_ROOT must point
# at the folder that contains checkpoint.py, main.py, model.py (run Cell 2 first).
import os
import sys
import tempfile
import atexit
from pathlib import Path


def _dir_has_checkpoint_py(d: Path) -> bool:
    """Colab/Linux is case-sensitive; Windows uploads may use Checkpoint.PY etc."""
    if not d.is_dir():
        return False
    direct = d / "checkpoint.py"
    if direct.is_file():
        return True
    try:
        for f in d.iterdir():
            if f.is_file() and f.name.lower() == "checkpoint.py":
                return True
    except OSError:
        pass
    return False


def resolve_signvlm_root(repo_root: str) -> Path:
    p = Path(repo_root).expanduser().resolve()
    if not p.exists():
        raise FileNotFoundError(
            f"REPO_ROOT path does not exist (mount Drive in Cell 2 first):\n  {p}"
        )

    candidates = []
    seen = set()

    def add(d: Path):
        d = d.resolve()
        if d not in seen and d.is_dir():
            seen.add(d)
            candidates.append(d)

    add(p)
    if p.name.lower() in ("notebooks", "notebook"):
        add(p.parent)
    try:
        for child in p.iterdir():
            if child.is_dir():
                add(child)
                for child2 in child.iterdir():
                    if child2.is_dir():
                        add(child2)
    except OSError:
        pass

    for d in candidates:
        if _dir_has_checkpoint_py(d):
            return d

    # What Colab actually sees (shortcuts / wrong folder are common on Drive)
    try:
        listing = ", ".join(sorted(os.listdir(p))[:50])
    except OSError as e:
        listing = f"(cannot list: {e})"
    raise FileNotFoundError(
        "Could not find checkpoint.py in REPO_ROOT or its immediate subfolders.\n"
        f"  REPO_ROOT: {p}\n"
        f"  Contents: {listing}\n"
        "Fix: set REPO_ROOT in Cell 2 to the folder that *directly* contains checkpoint.py "
        "(if you unzipped the repo, you may have an extra nested folder — point REPO_ROOT there).\n"
        "Google Drive shortcuts sometimes do not mount as real folders; copy the real folder.\n"
        "Or clone: !git clone <URL> \"/content/drive/MyDrive/FYP_DATA/Urdu-Sign-Language-Recognition-using-SignVLM\""
    )


REPO = resolve_signvlm_root(REPO_ROOT)
REPO_ROOT = str(REPO)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

os.chdir(REPO_ROOT)

import torch
import torch.distributed as dist

if not dist.is_initialized():
    _store_path = os.path.join(tempfile.gettempdir(), f"signvlm_store_{os.getpid()}")
    _store = dist.FileStore(_store_path, 1)
    dist.init_process_group(backend="gloo", store=_store, rank=0, world_size=1)
    atexit.register(lambda p=_store_path: os.remove(p) if os.path.exists(p) else None)

import checkpoint
import video_dataset
from model import EVLTransformer
from weight_loaders import weight_loader_fn_dict
from vision_transformer import vit_presets

print("Repo (sys.path):", REPO_ROOT)
print("CUDA:", torch.cuda.is_available())

FileNotFoundError: Could not find checkpoint.py in REPO_ROOT or its immediate subfolders.
  REPO_ROOT: /content/drive/MyDrive/FYP_DATA/Urdu-Sign-Language-Recognition-using-SignVLM
  Contents: CLIP_Weights
Fix: set REPO_ROOT in Cell 2 to the folder that *directly* contains checkpoint.py (if you unzipped the repo, you may have an extra nested folder — point REPO_ROOT there).
Google Drive shortcuts sometimes do not mount as real folders; copy the real folder.
Or clone: !git clone <URL> "/content/drive/MyDrive/FYP_DATA/Urdu-Sign-Language-Recognition-using-SignVLM"

### Confusion matrix & F1 helpers

`collect_predictions` matches `main.evaluate` (batch size 1, multi-view softmax mean). `print_confusion_and_f1` prints macro/weighted/micro F1, `classification_report`, and the confusion matrix (or saves `.npy` when `num_classes` is large).

In [ ]:
# Metrics (same inference as main.evaluate)
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score


def collect_predictions(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for data, labels in loader:
            data, labels = data.cuda(), labels.cuda()
            assert data.size(0) == 1
            if data.ndim == 6:
                data = data[0]
            logits = model(data)
            scores = logits.softmax(dim=-1).mean(dim=0)
            y_pred.append(int(scores.argmax().cpu()))
            y_true.append(int(labels.view(-1)[0].cpu()))
    return np.array(y_true), np.array(y_pred)


def print_confusion_and_f1(y_true, y_pred, num_classes, class_names, title, save_dir=None, max_print_classes=35):
    labels_idx = np.arange(num_classes)
    cm = confusion_matrix(y_true, y_pred, labels=labels_idx)
    names = [class_names[i] if i < len(class_names) else str(i) for i in range(num_classes)]
    print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")
    print(f"F1 (macro):     {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"F1 (weighted):  {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"F1 (micro):     {f1_score(y_true, y_pred, average='micro', zero_division=0):.4f}")
    print("\nClassification report:")
    print(
        classification_report(
            y_true, y_pred, labels=labels_idx, target_names=names, zero_division=0, digits=4
        )
    )
    if num_classes <= max_print_classes:
        print("\nConfusion matrix (rows=true class, cols=predicted class):")
        print(cm)
    else:
        print(f"\nConfusion matrix shape {cm.shape} (too large for full print).")
        if save_dir:
            import os
            safe = title.lower().replace(" ", "_").replace("/", "_")
            p = os.path.join(save_dir, f"{safe}_confusion_matrix.npy")
            np.save(p, cm)
            print(f"Saved full matrix: {p}")
        h = min(12, cm.shape[0])
        print(f"Top-left {h}x{h} block:\n{cm[:h, :h]}")

## Cell 4: Data preparation — TSV lists + loaders

- Builds a **class name → id** map from sorted folder names under `train_data` (ids `0 .. num_classes-1`).
- Writes `train.tsv`, `val.tsv`, `test.tsv`, `unseen.tsv` with lines `relative_path<TAB>label`, paths relative to each split root (same convention as `prepare_psl_splits.py` when `data_root` is the split folder).
- **Videos:** `frames_available=0` (PyAV). For **pre-extracted frames**, set `FRAMES_AVAILABLE = 1` and use the repo’s `tools/extract_frames_signvlm.py` first; lists must use virtual `.mp4` paths as in `SIGNVLM_TRAINING_NOTES.md`.
- Adjust **`NUM_CLASSES`** if you use a fixed label map file instead (must match folder names / ids).

In [ ]:
# Cell 4: Build label map and TSV lists; create train/val DataLoaders via repo helpers
from pathlib import Path
import argparse
import json

FRAMES_AVAILABLE = 0  # 1 if using pre-extracted frames (see SIGNVLM_TRAINING_NOTES.md)


def sorted_label_dirs(root: Path):
    return sorted([p for p in root.iterdir() if p.is_dir()], key=lambda p: p.name.casefold())


def build_label_map_from_train(train_root: Path):
    labels = [d.name for d in sorted_label_dirs(train_root)]
    return {name: i for i, name in enumerate(labels)}, labels


def iter_videos(label_dir: Path):
    vids = sorted(label_dir.glob("*.mp4"), key=lambda p: (not p.stem.isdigit(), int(p.stem) if p.stem.isdigit() else p.stem))
    for v in vids:
        yield v


def write_split_tsv(split_root: Path, rel_prefix: Path, name_to_id: dict, out_path: Path):
    n = 0
    with open(out_path, "w", encoding="utf-8") as f:
        for label_dir in sorted_label_dirs(split_root):
            name = label_dir.name
            if name not in name_to_id:
                raise ValueError(f"Unknown label folder {name!r} under {split_root}; not in train label map")
            lid = name_to_id[name]
            for vid in iter_videos(label_dir):
                rel = (rel_prefix / name / vid.name).as_posix()
                f.write(f"{rel}\t{lid}\n")
                n += 1
    return n


train_root = Path(TRAIN_DIR)
name_to_id, class_names = build_label_map_from_train(train_root)
NUM_CLASSES = len(name_to_id)

label_json = Path(LIST_DIR) / "label_map_auto.json"
with open(label_json, "w", encoding="utf-8") as f:
    json.dump({str(i): class_names[i] for i in range(len(class_names))}, f, ensure_ascii=False, indent=2)
print(f"num_classes={NUM_CLASSES}, wrote {label_json}")

train_tsv = Path(LIST_DIR) / "train.tsv"
val_tsv = Path(LIST_DIR) / "val.tsv"
test_tsv = Path(LIST_DIR) / "test.tsv"
unseen_tsv = Path(LIST_DIR) / "unseen.tsv"

n_train = write_split_tsv(train_root, Path("."), name_to_id, train_tsv)
n_val = write_split_tsv(Path(VAL_DIR), Path("."), name_to_id, val_tsv)
n_test = write_split_tsv(Path(TEST_DIR), Path("."), name_to_id, test_tsv)
n_unseen = write_split_tsv(Path(UNSEEN_DIR), Path("."), name_to_id, unseen_tsv)
print(f"lines: train={n_train} val={n_val} test={n_test} unseen={n_unseen}")

# Namespace with fields expected by video_dataset + main (PSL-style hyperparameters from SIGNVLM_TRAINING_NOTES)
args = argparse.Namespace(
    frames_available=FRAMES_AVAILABLE,
    train_list_path=str(train_tsv),
    val_list_path=str(val_tsv),
    train_data_root=TRAIN_DIR,
    val_data_root=VAL_DIR,
    data_root="",
    batch_size=8,
    n_shots=-1,
    num_spatial_views=1,
    num_temporal_views=3,
    num_frames=16,
    sampling_rate=4,
    tsn_sampling=False,
    spatial_size=224,
    mean=[0.48145466, 0.4578275, 0.40821073],
    std=[0.26862954, 0.26130258, 0.27577711],
    num_workers=4,
    dummy_dataset=False,
    auto_augment="rand-m7-n4-mstd0.5-inc1",
    interpolation="bicubic",
    mirror=True,
    num_steps=10000,
    checkpoint_dir=CHECKPOINT_DIR,
    auto_resume=False,
    resume_path=None,
    pretrain=None,
)

train_loader = video_dataset.create_train_loader(args, resume_step=0)
val_loader = video_dataset.create_val_loader(args)
print("train_loader steps:", len(train_loader), "expected:", args.num_steps)
print("val samples:", len(val_loader.dataset))


def make_eval_split_loader(tsv, root):
    """Eval-style loader (multi-view), same settings as val — for train/test/unseen metrics."""
    ns = argparse.Namespace(
        frames_available=args.frames_available,
        val_list_path=str(tsv),
        train_list_path=str(train_tsv),
        train_data_root=TRAIN_DIR,
        val_data_root=root,
        data_root="",
        batch_size=args.batch_size,
        n_shots=-1,
        num_spatial_views=1,
        num_temporal_views=3,
        num_frames=args.num_frames,
        sampling_rate=args.sampling_rate,
        tsn_sampling=False,
        spatial_size=args.spatial_size,
        mean=args.mean,
        std=args.std,
        num_workers=args.num_workers,
        dummy_dataset=False,
    )
    return video_dataset.create_val_loader(ns)

## Cell 5: Model — `EVLTransformer` (signVLM)

Matches **PSL** script settings: **ViT-L/14-lnpre**, decoder **4×1024**, **16 heads**, temporal conv / pos / cross-attention enabled (`docs/signVLMPipeline.md`). Backbone defaults to **frozen fp16** unless you set `finetune_backbone=True`.

In [ ]:
# Cell 5: EVLTransformer (ViT-L/14 + decoder)
finetune_backbone = False  # set True for end-to-end (see docs/signVLMPipeline.md)
fp16 = True

model = EVLTransformer(
    backbone_name="ViT-L/14-lnpre",
    backbone_type="clip",
    backbone_path=BACKBONE_PATH,
    backbone_mode="finetune" if finetune_backbone else ("freeze_fp16" if fp16 else "freeze_fp32"),
    decoder_num_layers=4,
    decoder_qkv_dim=1024,
    decoder_num_heads=16,
    decoder_mlp_factor=4.0,
    num_classes=NUM_CLASSES,
    enable_temporal_conv=True,
    enable_temporal_pos_embed=True,
    enable_temporal_cross_attention=True,
    cls_dropout=0.5,
    decoder_mlp_dropout=0.5,
    num_frames=args.num_frames,
)
model.cuda()
print(model.__class__.__name__, "num_classes:", NUM_CLASSES)

## Cell 6: Training configuration

- **Optimizer:** AdamW, `lr=4e-5`, `weight_decay=0.05` (`main.py` / notes).
- **Scheduler:** `CosineAnnealingLR(T_max=num_steps)`.
- **Loss:** `CrossEntropyLoss`.
- **AMP:** `GradScaler` + `autocast` (disable with `fp16=False`).
- **`batch_split=2`** matches the PSL shell script (gradient accumulation for memory).

In [ ]:
# Cell 6: Optimizer, scheduler, loss, AMP
lr = 4e-5
weight_decay = 0.05
batch_split = 2
num_steps = args.num_steps
save_freq = 2500
eval_freq = 2500
print_freq = 10

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
lr_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_steps)
loss_scaler = torch.amp.GradScaler("cuda", enabled=fp16)
criterion = torch.nn.CrossEntropyLoss()

args.save_freq = save_freq
args.eval_freq = eval_freq
args.print_freq = print_freq
args.batch_split = batch_split
args.fp16 = fp16
args.finetune_backbone = finetune_backbone

resume_step = checkpoint.resume_from_checkpoint(model, optimizer, lr_sched, loss_scaler, args)
print("resume_step:", resume_step)

## Cell 7: Training and validation loop

Mirrors `main.py`: micro-batching with `batch_split`, `GradScaler`, cosine LR step each iteration, periodic **`evaluate()`** (top-1 / top-5) on `val_loader`, then **confusion matrix + F1** on the same validation loader. After training finishes, runs the same metrics on the **train** split (eval-style loader). Checkpoints go to **Google Drive** (`CHECKPOINT_DIR`).

CLI alternative: `python main.py --train_list_path ... --val_list_path ... --checkpoint_dir ...` (see `scripts/train_psl_vitl14_16f_dec4x1024_1shot.sh`).

In [ ]:
# Cell 7: Training + validation (same structure as main.py)
from datetime import datetime

import main as signvlm_main

train_loader = video_dataset.create_train_loader(args, resume_step=resume_step)
assert len(train_loader) == args.num_steps - resume_step

model.train()
train_st = datetime.now()
batch_st = datetime.now()

for i, (data, labels) in enumerate(train_loader, resume_step):
    data, labels = data.cuda(), labels.cuda()
    data_ed = datetime.now()
    optimizer.zero_grad(set_to_none=True)

    assert data.size(0) % args.batch_split == 0
    split_size = data.size(0) // args.batch_split
    hit1, hit5, loss_value = 0, 0, 0.0

    for j in range(args.batch_split):
        sl = slice(split_size * j, split_size * (j + 1))
        data_slice, labels_slice = data[sl], labels[sl]
        with torch.cuda.amp.autocast(args.fp16):
            logits = model(data_slice)
            loss = criterion(logits, labels_slice)
        if labels.dtype == torch.long:
            hit1 += (logits.topk(1, dim=1)[1] == labels_slice.view(-1, 1)).sum().item()
            hit5 += (logits.topk(5, dim=1)[1] == labels_slice.view(-1, 1)).sum().item()
        loss_value += loss.item() / args.batch_split
        loss_scaler.scale(loss / args.batch_split).backward()

    loss_scaler.step(optimizer)
    loss_scaler.update()
    lr_sched.step()
    batch_ed = datetime.now()

    if i % args.print_freq == 0:
        sync_tensor = torch.tensor([loss_value, hit1 / data.size(0), hit5 / data.size(0)], device="cuda")
        dist.all_reduce(sync_tensor)
        sync_tensor = sync_tensor.cpu() / dist.get_world_size()
        loss_value, acc1, acc5 = sync_tensor.tolist()
        print(
            f"Step {i} batch {(batch_ed - batch_st).total_seconds():.3f}s "
            f"lr {optimizer.param_groups[0]['lr']:.6f} loss {loss_value:.6f} "
            f"acc1 {acc1 * 100:.2f}% acc5 {acc5 * 100:.2f}%"
        )

    if (i + 1) % args.save_freq == 0:
        checkpoint.save_checkpoint(model, optimizer, lr_sched, loss_scaler, i + 1, args)

    if (i + 1) % args.eval_freq == 0:
        print("Eval at step", i + 1)
        model.eval()
        signvlm_main.evaluate(model, val_loader)
        yt, yp = collect_predictions(model, val_loader)
        print_confusion_and_f1(yt, yp, NUM_CLASSES, class_names, "validation", save_dir=LIST_DIR)
        model.train()

    batch_st = datetime.now()

checkpoint.save_checkpoint(model, optimizer, lr_sched, loss_scaler, num_steps, args)
print("Training done:", datetime.now() - train_st)

# Train split: confusion matrix + F1 (eval-style loader, no augmentation)
model.eval()
train_eval_loader = make_eval_split_loader(train_tsv, TRAIN_DIR)
yt_tr, yp_tr = collect_predictions(model, train_eval_loader)
print_confusion_and_f1(yt_tr, yp_tr, NUM_CLASSES, class_names, "train", save_dir=LIST_DIR)

## Cell 8: Evaluation on `test_data` and `Unseen_data`

Runs `main.evaluate` (top-1 / top-5), then **confusion matrix + macro/weighted/micro F1 + per-class report** for **test** and **unseen** (same inference as validation). Latest `checkpoint-*.pth` under `CHECKPOINT_DIR` is loaded unless you change `EVAL_CKPT`.

In [ ]:
# Cell 8: Eval test split and unseen split
import os
import main as signvlm_main

def latest_checkpoint(ckpt_dir):
    files = [f for f in os.listdir(ckpt_dir) if f.startswith("checkpoint-") and f.endswith(".pth")]
    if not files:
        return None
    best = max(files, key=lambda f: int(f.replace("checkpoint-", "").replace(".pth", "")))
    return os.path.join(ckpt_dir, best)

EVAL_CKPT = latest_checkpoint(CHECKPOINT_DIR) or os.path.join(CHECKPOINT_DIR, "checkpoint-10000.pth")

eval_args = argparse.Namespace(
    frames_available=args.frames_available,
    val_list_path=str(test_tsv),
    train_list_path=str(train_tsv),
    train_data_root=TRAIN_DIR,
    val_data_root=TEST_DIR,
    data_root="",
    batch_size=args.batch_size,
    n_shots=-1,
    num_spatial_views=1,
    num_temporal_views=3,
    num_frames=args.num_frames,
    sampling_rate=args.sampling_rate,
    tsn_sampling=False,
    spatial_size=args.spatial_size,
    mean=args.mean,
    std=args.std,
    num_workers=args.num_workers,
    dummy_dataset=False,
    auto_augment=None,
    interpolation="bicubic",
    mirror=False,
    checkpoint_dir=None,
    auto_resume=False,
    resume_path=None,
    pretrain=None,
)

model.eval()
ckpt = torch.load(EVAL_CKPT, map_location="cpu")
model.load_state_dict(ckpt["model"], strict=True)
print("Loaded:", EVAL_CKPT)

print("=== test_data ===")
test_loader = video_dataset.create_val_loader(eval_args)
signvlm_main.evaluate(model, test_loader)
yt_te, yp_te = collect_predictions(model, test_loader)
print_confusion_and_f1(yt_te, yp_te, NUM_CLASSES, class_names, "test", save_dir=LIST_DIR)

eval_args.val_list_path = str(unseen_tsv)
eval_args.val_data_root = UNSEEN_DIR
print("=== Unseen_data ===")
unseen_loader = video_dataset.create_val_loader(eval_args)
signvlm_main.evaluate(model, unseen_loader)
yt_un, yp_un = collect_predictions(model, unseen_loader)
print_confusion_and_f1(yt_un, yp_un, NUM_CLASSES, class_names, "unseen", save_dir=LIST_DIR)